**`02_show_footprints_fast`**

Tutorial for large-scale (1-5M polygon) footprint/building visualization: an
interactive GPU-accelerated map (Lonboard/deck.gl) and a CPU-only raster
fallback (Datashader). Requires the `viz-fast` extra
(`pip install -e ".[viz-fast]"` or `conda env update -f environment.yml`).

There is no reliable way for a Jupyter kernel to detect whether the browser
has WebGL2 available, so the choice between the two tracks below is left to
you, the caller — not automatic.

In [ ]:
from openplaces.viz import show_entities_interactive, show_entities_raster

# Harmonized (evidence-only) national footprint spine. Used here rather than
# the curated `US_footprint-cheer-2026` deliverable because, at the time of
# writing, only a couple of NC counties have been reprocessed under CHEER's
# current `save_to: share, combined: true` output location -- the recipe's
# older `core/`-directory output is still on disk for most counties, but
# `get_entities` resolves paths from the *current* recipe config, so it
# can't see those legacy files. The spine recipe's own `save_to: core`
# hasn't changed, so it resolves the full set of already-processed counties.
RECIPE = 'US_footprint-spine-2026'
ONE_COUNTY = 'US-NC-BA'  # Beaufort County, NC
STATE = 'US-NC'  # North Carolina; ~44 of 100 counties processed so far

# Track 1 — interactive GPU rendering (Lonboard/deck.gl)

Uses `SolidPolygonLayer` (not the composite `PolygonLayer`, whose extra
outline draw pass is unnecessary overhead at this scale). The map object
renders automatically below and stays interactive: pan, zoom, tilt, rotate.

In [ ]:
# One county, colored by occupancy_type (canonical footprint classification;
# matches a built-in palette in openplaces.viz.colors -- openplaces_group_parcel
# is a raw parcel-evidence column and not a good coloring choice).
show_entities_interactive(RECIPE, admin_id=ONE_COUNTY, color_by='occupancy_type')

## Viewport-bounded loading (`bbox`)

`get_entities` (and these two functions, which call it) accept a `bbox` in
EPSG:4326, exploiting the covering-bbox column written at save time for
Parquet-level predicate pushdown. Files with no overlap contribute zero
rows, so loading a bbox across many admin-unit files stays cheap — the
mechanism behind progressive/viewport-bound loading, without a new query
engine.

The full state (`missing='ignore'` renders whatever counties are already
processed) is close to the 1-5M polygon scale this whole architecture
targets.

In [ ]:
# A bbox around just the coastal half of one county -- loads a subset of a
# single file via Parquet-level predicate pushdown.
coastal_bbox = (-76.75, 35.35, -76.55, 35.55)
show_entities_interactive(RECIPE, admin_id=ONE_COUNTY, bbox=coastal_bbox)

In [ ]:
# Full state: ~2.7M polygons across the ~44 already-processed counties.
# missing='ignore' skips the ~56 counties not yet processed under this
# recipe rather than raising.
show_entities_interactive(
    RECIPE, admin_id=STATE, color_by='occupancy_type', missing='ignore'
)

# Track 2 — CPU raster fallback (Datashader)

For headless/CPU-only environments where Track 1's WebGL2 client requirement
can't be met. Rasterizes server-side to a fixed-resolution image; no pan/zoom
without re-running the cell. Reprojects to EPSG:3857 before rasterizing
(`Canvas.polygons` has no notion of projection, and NC spans enough latitude
that leaving data in raw lon/lat visibly distorts aspect ratio).

In [ ]:
# One county, plain density heatmap (no color_by).
show_entities_raster(RECIPE, admin_id=ONE_COUNTY)

In [ ]:
# Full state, colored by occupancy_type -- same ~2.7M-polygon scale as the
# interactive map above, rasterized in a single pass (no Dask; see
# openplaces.viz.raster's docstring for when chunked aggregation would
# actually be needed).
show_entities_raster(
    RECIPE,
    admin_id=STATE,
    color_by='occupancy_type',
    missing='ignore',
    plot_width=1200,
    plot_height=800,
)

## Notes

- `simplified=True` requests geometry from a `_geo_simplified.parquet`
  sidecar (faster transport/render, lower vertex fidelity). It's currently
  only written for admin geometries, not footprint/building recipes, so
  `show_entities_raster` defaults to `simplified=False` and this notebook
  doesn't turn it on — trying it today against a footprint/building recipe
  will raise `FileNotFoundError`.
- `color_by` values without a match in `openplaces.viz.colors.CATEGORY_COLORS`
  (checked via `match_palette`) fall back to a default grey/green rather than
  erroring — e.g. raw evidence columns like `openplaces_group_parcel`
  (attributed from parcels onto footprint geometries, not itself a
  registered palette) fall back this way; `occupancy_type` above is the
  canonical footprint classification and has its own registered palette.
- `US_footprint-cheer-2026` (the terminal, curated deliverable) is used
  nowhere in this notebook because its current `save_to: share,
  combined: true` output location only has ~2 of NC's counties processed so
  far; swap `RECIPE` once more counties have been reprocessed there.